# Install libs

In [ ]:
# %pip install zss
# %pip install pandas

# Import packages and define variables

In [ ]:
import os
import json
import re
import functools
import pandas as pd
from zss import distance, simple_distance, Node
from zss.simple_tree import Node

# Get all graphs

In [ ]:
# Define the input folder path (using absolute path)
input_folder = "/home/leo/github/ils-clustering-hmd/data/clustering/metrics/input"

# Initialize the main results dictionary
algorithm_results = {}

def extract_project_name(filename):
    """Extract project name from filename like '_b01-jgit-http-6.8.0 61C.odem.json'"""
    # Remove the leading '_b01-' and the ' XC.odem...' part
    match = re.match(r'_b01-(.+?) \d+C\.odem', filename)
    if match:
        return match.group(1)
    return None

def load_algorithm_results(algorithm_type, folder_path):
    """Load results for a specific algorithm type"""
    results = {}
    
    if not os.path.exists(folder_path):
        print(f"Warning: Folder {folder_path} does not exist")
        return results
    
    for filename in os.listdir(folder_path):
        if not filename.endswith('.json'):
            continue
            
        project_name = extract_project_name(filename)
        if not project_name:
            print(f"Warning: Could not extract project name from {filename}")
            continue
        
        # Read the JSON file
        file_path = os.path.join(folder_path, filename)
        try:
            with open(file_path, 'r') as f:
                data = json.load(f)
            
            # For CRG, extract standard deviation from filename
            if algorithm_type == 'crg':
                stddev_match = re.search(r'_stddev_(\d+)\.json$', filename)
                if stddev_match:
                    stddev = stddev_match.group(1)
                    if project_name not in results:
                        results[project_name] = {}
                    results[project_name][stddev] = data
                else:
                    print(f"Warning: Could not extract stddev from CRG file {filename}")
            # For MQ, extract iteration from filename (non-deterministic algorithm)
            elif algorithm_type == 'mq':
                iteration_match = re.search(r'_iteration_(\d+)\.json$', filename)
                if iteration_match:
                    iteration = iteration_match.group(1)
                    if project_name not in results:
                        results[project_name] = {}
                    results[project_name][iteration] = data
                else:
                    print(f"Warning: Could not extract iteration from MQ file {filename}")
            else:
                # For original and hmd, just store the data directly
                results[project_name] = data
                
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    
    return results

# Load results for each algorithm
print("Loading algorithm results...")

# Load original algorithm results
original_results = load_algorithm_results('original', os.path.join(input_folder, 'results_original'))
print(f"Loaded {len(original_results)} original algorithm results")

# Load HMD algorithm results  
hmd_results = load_algorithm_results('hmd', os.path.join(input_folder, 'results_hmd'))
print(f"Loaded {len(hmd_results)} HMD algorithm results")

# Load CRG algorithm results
crg_results = load_algorithm_results('crg', os.path.join(input_folder, 'results_crg'))
print(f"Loaded {len(crg_results)} CRG algorithm results")

# Load MQ algorithm results (non-deterministic, multiple iterations)
mq_results = load_algorithm_results('mq', os.path.join(input_folder, 'results_mq'))
print(f"Loaded {len(mq_results)} MQ algorithm results")

# Organize all results in a unified structure
# Structure: results[project_name] = {'original': data, 'hmd': data, 'crg': {'25': data, '50': data, '75': data}, 'mq': {'1': data, '2': data, ...}}

# Get all unique project names
all_projects = set()
all_projects.update(original_results.keys())
all_projects.update(hmd_results.keys()) 
all_projects.update(crg_results.keys())
all_projects.update(mq_results.keys())

print(f"Found {len(all_projects)} unique projects")

# Create unified results structure
unified_results = {}
for project in sorted(all_projects):
    unified_results[project] = {
        'original': original_results.get(project),
        'hmd': hmd_results.get(project),
        'crg': crg_results.get(project, {}),
        'mq': mq_results.get(project, {})
    }

In [ ]:
# 1. Create comparison pairs for analysis
print(f"\n=== Setting up comparison pairs ===")

# Initialize comparison structure
comparison_pairs = []

for project in unified_results:
    project_data = unified_results[project]
    
    # Only process projects that have all required data
    if (project_data['original'] and 
        project_data['hmd'] and 
        '25' in project_data['crg'] and 
        '50' in project_data['crg'] and 
        '75' in project_data['crg'] and
        project_data['mq']):  # At least one MQ iteration
        
        # Create comparison pairs: original vs each other algorithm
        base_comparisons = [
            {
                'project': project,
                'baseline': 'original',
                'baseline_data': project_data['original'],
                'compare_to': 'hmd',
                'compare_data': project_data['hmd']
            },
            {
                'project': project,
                'baseline': 'original', 
                'baseline_data': project_data['original'],
                'compare_to': 'crg_stddev_25',
                'compare_data': project_data['crg']['25']
            },
            {
                'project': project,
                'baseline': 'original',
                'baseline_data': project_data['original'], 
                'compare_to': 'crg_stddev_50',
                'compare_data': project_data['crg']['50']
            },
            {
                'project': project,
                'baseline': 'original',
                'baseline_data': project_data['original'],
                'compare_to': 'crg_stddev_75', 
                'compare_data': project_data['crg']['75']
            }
        ]
        
        # Add MQ iterations
        mq_data = project_data['mq']
        for iteration in sorted(mq_data.keys(), key=int):
            base_comparisons.append({
                'project': project,
                'baseline': 'original',
                'baseline_data': project_data['original'],
                'compare_to': f'mq_iteration_{iteration}',
                'compare_data': mq_data[iteration]
            })
        
        comparison_pairs.extend(base_comparisons)

print(f"Created {len(comparison_pairs)} comparison pairs")

In [ ]:
# Verify all 29 projects are included in comparison_pairs
print("=== Verification: All Projects in Comparison Pairs ===")

# Extract unique projects from comparison_pairs
projects_in_comparison = set(pair['project'] for pair in comparison_pairs)
print(f"Number of projects in comparison_pairs: {len(projects_in_comparison)}")

# Show all projects included
print(f"\nAll {len(projects_in_comparison)} projects included in comparisons:")
for i, project in enumerate(sorted(projects_in_comparison), 1):
    print(f"{i:2d}. {project}")

# Verify the math: should be 29 projects × 4 comparisons = 116 total pairs
expected_total = len(projects_in_comparison) * 4
print(f"\nVerification:")
print(f"  Projects with complete data: {len(projects_in_comparison)}")
print(f"  Comparisons per project: 4 (original vs hmd, crg_25, crg_50, crg_75)")
print(f"  Expected total pairs: {len(projects_in_comparison)} × 4 = {expected_total}")
print(f"  Actual total pairs: {len(comparison_pairs)}")
print(f"  ✓ Match: {expected_total == len(comparison_pairs)}")

# Show which projects were excluded (if any)
all_projects_with_original_and_hmd = set()
for project, data in unified_results.items():
    if data['original'] and data['hmd']:
        all_projects_with_original_and_hmd.add(project)

excluded_projects = all_projects_with_original_and_hmd - projects_in_comparison
if excluded_projects:
    print(f"\nProjects excluded (missing CRG data): {len(excluded_projects)}")
    for project in sorted(excluded_projects):
        crg_data = unified_results[project]['crg']
        missing_stddevs = []
        for stddev in ['25', '50', '75']:
            if stddev not in crg_data:
                missing_stddevs.append(stddev)
        print(f"  - {project}: missing CRG stddev {', '.join(missing_stddevs)}")
else:
    print(f"\n✓ No projects excluded - all projects with original+hmd also have complete CRG data!")

print(f"\n Ready for analysis with all {len(projects_in_comparison)} complete datasets!")

# Define functions

In [ ]:
def define_tree_zss(parent, root, nodes, prefix, level, root_nodes):
    # print("root") if root == None else print(prefix + root['data']['id'])
    if parent == None:
        parent = Node('root')
    for node in nodes:
        node_name = node['data']['id']
        new_node = Node(node_name)
        parent.addkid(new_node)
        children = [x for x in root_nodes if 'parent' in x['data'] and x['data']['parent'] == node_name]
        define_tree_zss(new_node, node, children, '--' * level, level + 1, root_nodes)
    return (parent)

In [ ]:
def zero_or_one_distance(a, b):
    if a == b:
        return 0
    else:
        return 1

def default_tree_size(tree,get_children):
    size = 1
    children = get_children(tree)
    if children:
        size = size + sum(default_tree_size(child,get_children) for child in children)
    return size

def _compute_normalized_distance(edit_distance,alpha,size_A,size_B):
    #return edit_distance / max(size_A, size_B)
    return edit_distance / (size_A + size_B)
    #return (2*edit_distance)/(alpha*(size_A+size_B)+edit_distance)

def normalized_simple_distance(A,B,get_children=Node.get_children,alpha=1,get_tree_size=None,return_operations=False,**kwargs):
    if get_tree_size is None:
        get_tree_size = functools.partial(default_tree_size,get_children=get_children)
    if return_operations:
        edit_distance, operations = simple_distance(A,B,get_children=get_children,**kwargs)
    else:
        edit_distance = simple_distance(A,B,get_children=get_children,**kwargs)
    d = _compute_normalized_distance(edit_distance,alpha,get_tree_size(A),get_tree_size(B))
    if return_operations:
        return d, operations
    else:
        return d

def normalized_distance(A,B,get_children,alpha=1,get_tree_size=None,return_operations=False,**kwargs):
    if get_tree_size is None:
        get_tree_size = functools.partial(default_tree_size,get_children=get_children)
    if return_operations:
        edit_distance, operations = distance(A,B,get_children,**kwargs)
    else:
        edit_distance = distance(A,B,get_children,**kwargs)
    d = _compute_normalized_distance(edit_distance,alpha,get_tree_size(A),get_tree_size(B))
    if return_operations:
        return d, operations
    else:
        return d

# Calculate distance

In [ ]:
# Updated calculation using our organized data structure
results = []

print(f"Starting distance calculations for {len(comparison_pairs)} comparison pairs...")

for i, pair in enumerate(comparison_pairs):
    project = pair['project']
    baseline_type = pair['baseline']
    compare_type = pair['compare_to']
    baseline_data = pair['baseline_data']
    compare_data = pair['compare_data']
    
    # Progress indicator
    if (i + 1) % 20 == 0:
        print(f"Progress: {i + 1}/{len(comparison_pairs)} comparisons completed")
    
    try:
        # Extract nodes from the JSON data for both algorithms
        baseline_nodes = [x for x in baseline_data if x['group'] == 'nodes']
        compare_nodes = [x for x in compare_data if x['group'] == 'nodes']
        
        # Build trees using our define_tree_zss function
        T1 = define_tree_zss(None, None, 
                            [x for x in baseline_nodes if 'parent' not in x['data'] or x['data']['parent'] == 'root' or x['data']['parent'] == None], 
                            '--', 1, baseline_nodes)
        T2 = define_tree_zss(None, None, 
                            [x for x in compare_nodes if 'parent' not in x['data'] or x['data']['parent'] == 'root' or x['data']['parent'] == None], 
                            '--', 1, compare_nodes)
        
        # Calculate distances
        (result, operations) = simple_distance(T1, T2, return_operations=True)
        nresult = normalized_simple_distance(T1, T2)
        
        # Store results
        results.append({
            'project': project,
            'baseline_type': baseline_type,
            'compare_type': compare_type,
            'distance': result,
            'normalized_distance': nresult,
            'operations': operations
        })
        
    except Exception as e:
        print(f"Error calculating distance for {project} ({baseline_type} vs {compare_type}): {e}")
        # Add a record with error for debugging
        results.append({
            'project': project,
            'baseline_type': baseline_type,
            'compare_type': compare_type,
            'distance': None,
            'normalized_distance': None,
            'operations': None,
            'error': str(e)
        })

print(f"\nCompleted distance calculations!")
print(f"Total results: {len(results)}")
print(f"Successful calculations: {len([r for r in results if r.get('distance') is not None])}")
print(f"Failed calculations: {len([r for r in results if r.get('distance') is None])}")

In [ ]:
# Convert results to a DataFrame and save to CSV
results_df = pd.DataFrame(results)

# Remove operations column for the main results (too large for CSV)
if 'operations' in results_df.columns:
    results_df_clean = results_df.drop(columns=['operations'])
else:
    results_df_clean = results_df.copy()

# Save the DataFrame to a CSV file
results_df_clean.to_csv('results.csv', index=False)


In [ ]:
# Export operations data for detailed analysis
if results and any(r.get('operations') for r in results):
    # Find first result with operations data
    first_with_ops = next((r for r in results if r.get('operations')), None)
    
    if first_with_ops:
        # Convert operations to DataFrame
        operations_df = pd.DataFrame(first_with_ops['operations'])
        
        # Add context information
        operations_df['project'] = first_with_ops['project']
        operations_df['baseline_type'] = first_with_ops['baseline_type'] 
        operations_df['compare_type'] = first_with_ops['compare_type']
        
        # Save operations to CSV
        operations_df.to_csv('operations.csv', index=False)
        
        print(f"Tree edit operations saved to 'operations.csv'")
        print(f"Operations from: {first_with_ops['project']} ({first_with_ops['baseline_type']} vs {first_with_ops['compare_type']})")
        print(f"Number of operations: {len(operations_df)}")
        
        # Show operation types
        if 'type' in operations_df.columns:
            print(f"\nOperation types:")
            print(operations_df['type'].value_counts())
    else:
        print("No operations data found in results")
        
    # Optionally, export all operations data (warning: can be very large)
    export_all = False  # Set to True if you want all operations data
    
    if export_all:
        all_operations = []
        for r in results:
            if r.get('operations'):
                for op in r['operations']:
                    op_record = op.copy()
                    op_record['project'] = r['project']
                    op_record['baseline_type'] = r['baseline_type']
                    op_record['compare_type'] = r['compare_type']
                    all_operations.append(op_record)
        
        if all_operations:
            all_ops_df = pd.DataFrame(all_operations)
            all_ops_df.to_csv('all_tree_edit_operations.csv', index=False)
            print(f"All operations data saved to 'all_tree_edit_operations.csv' ({len(all_operations)} total operations)")
else:
    print("No operations data available in results")

- ref: https://zhang-shasha.readthedocs.io/en/latest/